this notebook separates the sequences, which are processed by IgBLAST, by germline V gene and bioproject, and saved them in designated directories
<p>to do so, each sequence is assigned the original RIOT germline V gene annotation and bioproject by its sra identification

In [1]:
import polars as pl
import numpy as np
from pathlib import Path

# loading the original paired AbNGS database with annotations and bioprojects

In [ ]:

PROJECT_DIR = Path("INSERT_PATH_TO_DATABASE")

airr = pl.scan_csv(
    str(PROJECT_DIR / "bioproject=*" / "*.csv"),
    separator="\t",
    infer_schema_length=1000,
    low_memory=True,
    schema_overrides={"sra": pl.String},
)


# assign bioprojects
root = PROJECT_DIR

lookup_dir = root / "_sra_lookup_parts"

lookup_dir.mkdir(exist_ok=True)

for project_dir in root.glob("bioproject=*"): #loops bioprojects from database

    if project_dir == lookup_dir:
        continue

    bioproject = project_dir.name.removeprefix("bioproject=") #example name bioproject directory: bioproject=PRJNA782883

    files = [str(f) for f in project_dir.glob("*.csv")]

    if not files:
        continue

    (
        pl.scan_csv(files, separator='\t')
        .select(
            pl.col("sra").cast(pl.Utf8).alias("sra_heavy")
        )
        .drop_nulls()
        .unique()
        .with_columns(
            pl.lit(bioproject).alias("bioproject")
        )
        .sink_parquet(
            lookup_dir / f"{bioproject}.parquet"
        )
    )

# heavy chains

In [ ]:
lf = pl.scan_csv(
    "sequences_heavy_db-pass.tsv",
    separator="\t",
)
lf = lf.with_columns(pl.col('sequence_id').str.split('_').list.get(2).alias('sra'))


In [ ]:
# assign bioprojects
root = PROJECT_DIR

lookup_dir = root / "_sra_lookup_parts"

# assigns every sra the corresponding bioproject
sra_lookup = (
    pl.scan_parquet(str(lookup_dir / "*.parquet"))
    .unique(subset=["sra_heavy"]).rename({"sra_heavy": "sra"})
)

# assigns to every sequence a bioproject, based on the sra
lf_bp = (
    lf
    .join(
        sra_lookup,
        on="sra",
        how="left"
    )
)

In [ ]:
# for all heavy chain v genes separated additionally by project

# Build the join key once
lf_bp = lf_bp.drop("v_call", strict=False)

# Filter early (only heavy chains) and select only the required columns --> v_call: original RIOT annotation from database, sequence_id: to identify sequences
hv_v_calls = (
    airr
    .filter((pl.col("v_call").str.contains("HV")) & (pl.col("v_call").str.contains(r"IGHV\d")))
    .with_columns((pl.concat_str(["sequence_id", "sra"], separator="_").alias("sequence_id"))
    , (pl.col('v_call').str.split('*').list.get(0).alias('v_call')))
    .select(["sequence_id", "v_call"])
)

# Join
lf_bp = lf_bp.join(hv_v_calls, on="sequence_id", how="left")
base_dir = Path("./by_vgene_project") 
base_dir.mkdir(parents=True, exist_ok=True)

# keep only sequences which have a germline alignment (after IgBLAST)
lf_filtered = (
    lf_bp
    .filter(pl.col("germline_alignment").is_not_null())
)

df = lf_filtered.collect(engine="streaming")

print(f"Loaded {df.height:,} sequences")

# Split into germline V gene and bioproject groups
groups = df.partition_by(
    ["v_call", "bioproject"],
    as_dict=True,
    maintain_order=False,
)

# Write each group without rerunning the query
for (v, bp), group_df in groups.items():
    output_dir = base_dir / f"OE_heavy_{v}_{bp}"
    output_file = output_dir / "sequences_heavy.tsv"

    if output_file.exists():
        continue

    output_dir.mkdir(parents=True, exist_ok=True)

    group_df.write_csv(
        output_file,
        separator="\t",
    )

    print(f"Saved {output_file} ({group_df.height:,} rows)")

del groups, df


# lambda

In [ ]:
lf = pl.scan_csv(
    "sequences_lambda_db-pass.tsv",
    separator="\t",
)
lf = lf.with_columns(pl.col('sequence_id').str.split('_').list.get(2).alias('sra'))

root = PROJECT_DIR

lookup_dir = root / "_sra_lookup_parts"
sra_lookup = (
    pl.scan_parquet(str(lookup_dir / "*.parquet"))
    .unique(subset=["sra_heavy"]).rename({"sra_heavy": "sra"})
)

lf_bp = (
    lf
    .join(
        sra_lookup,
        on="sra",
        how="left"
    )
)



In [ ]:

# for all lambda chain v genes separated additionally by project

# Build the join key once
lf_bp = lf_bp.drop("v_call", strict=False)

lv_v_calls = (
    airr
    .filter((pl.col("v_call").str.contains("LV")) & (pl.col("v_call").str.contains(r"IGLV\d")))
    .with_columns((pl.concat_str(["sequence_id", "sra"], separator="_").alias("sequence_id"))
    , (pl.col('v_call').str.split('*').list.get(0).alias('v_call')))
    .select(["sequence_id", "v_call"])
)


lf_bp = lf_bp.join(lv_v_calls, on="sequence_id", how="left")
base_dir = Path("by_vgene_project")
base_dir.mkdir(parents=True, exist_ok=True)

lf_filtered = (
    lf_bp
    .filter(pl.col("germline_alignment").is_not_null())
)

df = lf_filtered.collect(engine="streaming")

print(f"Loaded {df.height:,} sequences")

groups = df.partition_by(
    ["v_call", "bioproject"],
    as_dict=True,
    maintain_order=False,
)

for (v, bp), group_df in groups.items():
    output_dir = base_dir / f"OE_lambda_{v}_{bp}"
    output_file = output_dir / "sequences_lambda.tsv"

    if output_file.exists():
        continue

    output_dir.mkdir(parents=True, exist_ok=True)

    group_df.write_csv(
        output_file,
        separator="\t",
    )

    print(f"Saved {output_file} ({group_df.height:,} rows)")

del groups, df


# kappa

In [ ]:
lf = pl.scan_csv(
    "sequences_kappa_db-pass.tsv", 
    separator="\t",
)
lf = lf.with_columns(pl.col('sequence_id').str.split('_').list.get(2).alias('sra'))

root = PROJECT_DIR

lookup_dir = root / "_sra_lookup_parts"
sra_lookup = (
    pl.scan_parquet(str(lookup_dir / "*.parquet"))
    .unique(subset=["sra_heavy"]).rename({"sra_heavy": "sra"})
)

lf_bp = (
    lf
    .join(
        sra_lookup,
        on="sra",
        how="left"
    )
)



In [ ]:

# for all heavy chain v genes separated additionally by project

# Build the join key once
lf_bp = lf_bp.drop("v_call", strict=False)


kv_v_calls = (
    airr
    .filter((pl.col("v_call").str.contains("KV")) & (pl.col("v_call").str.contains(r"IGKV\d")))
    .with_columns((pl.concat_str(["sequence_id", "sra"], separator="_").alias("sequence_id"))
    , (pl.col('v_call').str.split('*').list.get(0).alias('v_call')))
    .select(["sequence_id", "v_call"])
)


lf_bp = lf_bp.join(kv_v_calls, on="sequence_id", how="left")
base_dir = Path("by_vgene_project") 
base_dir.mkdir(parents=True, exist_ok=True)

lf_filtered = (
    lf_bp
    .filter(pl.col("germline_alignment").is_not_null())
)

df = lf_filtered.collect(engine="streaming")

print(f"Loaded {df.height:,} sequences")

groups = df.partition_by(
    ["v_call", "bioproject"],
    as_dict=True,
    maintain_order=False,
)

for (v, bp), group_df in groups.items():
    output_dir = base_dir / f"OE_kappa_{v}_{bp}"
    output_file = output_dir / "sequences_kappa.tsv"

    if output_file.exists():
        continue

    output_dir.mkdir(parents=True, exist_ok=True)

    group_df.write_csv(
        output_file,
        separator="\t",
    )

    print(f"Saved {output_file} ({group_df.height:,} rows)")

del groups, df
